# 03 — Bus-Based Grid Network Build

**Purpose:** Build a bus-topology NetworkX graph of the US power grid.
Buses are derived by clustering transmission-line endpoints; power plants are
snapped to their nearest bus.

**Inputs:**
- `data/processed/power_plants.geojson` — EIA operating generators (notebook 01)
- `data/processed/osm_transmission_hv.geojson` — OSM HV transmission lines (notebook 04)

**Outputs:**
- `data/processed/grid_network.graphml` — NetworkX graph (GraphML)
- `data/processed/bus_locations.geojson` — Bus locations with attributes

In [4]:
import sys
from pathlib import Path
from collections import defaultdict

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
from shapely.geometry import Point

PROCESSED = PROJECT_ROOT / "data" / "processed"
print("Imports OK")

Imports OK


## Step 1 — Load and Validate

In [5]:
print("─" * 60)
print("STEP 1 — Load and validate")
print("─" * 60)

plants = gpd.read_file(PROCESSED / "power_plants.geojson")
lines  = gpd.read_file(PROCESSED / "osm_transmission_hv.geojson")

assert plants.crs.to_epsg() == 4326, f"plants CRS is {plants.crs}, expected EPSG:4326"
assert lines.crs.to_epsg()  == 4326, f"lines CRS is {lines.crs}, expected EPSG:4326"
print("CRS assertions passed (both EPSG:4326)")

print(f"\nplants  shape : {plants.shape}")
print(f"plants  cols  : {plants.columns.tolist()}")
print(f"\nlines   shape : {lines.shape}")
print(f"lines   cols  : {lines.columns.tolist()}")
print(f"\nGeometry types in lines:")
print(lines.geometry.geom_type.value_counts().to_string())
print(f"\nvoltage_kv range : {lines['voltage_kv'].min():.0f} – {lines['voltage_kv'].max():.0f} kV")

────────────────────────────────────────────────────────────
STEP 1 — Load and validate
────────────────────────────────────────────────────────────
CRS assertions passed (both EPSG:4326)

plants  shape : (15000, 25)
plants  cols  : ['period', 'stateid', 'stateName', 'sector', 'sectorName', 'entityid', 'entityName', 'plantid', 'plantName', 'generatorid', 'technology', 'energy_source_code', 'energy-source-desc', 'prime_mover_code', 'balancing_authority_code', 'balancing-authority-name', 'status', 'statusDescription', 'nameplate-capacity-mw', 'latitude', 'longitude', 'county', 'unit', 'nameplate-capacity-mw-units', 'geometry']

lines   shape : (168025, 6)
lines   cols  : ['osm_id', 'voltage_kv', 'name', 'operator', 'cables', 'geometry']

Geometry types in lines:
LineString    168025

voltage_kv range : 115 – 1333 kV


## Step 2 — Build Bus Topology

Strategy: extract the start and end point of every LineString, project to
EPSG:5070 (metric), then snap all endpoints to a 500 m grid.  Every distinct
grid cell becomes one bus.  The bus location is the mean of all endpoints that
fall in that cell.

In [6]:
print("─" * 60)
print("STEP 2 — Build bus topology")
print("─" * 60)

# ── Project lines to EPSG:5070 (Albers Equal Area, metres) ───────────────────
print("Projecting transmission lines to EPSG:5070...")
lines_proj = lines.to_crs(epsg=5070)

# ── Extract endpoints ─────────────────────────────────────────────────────────
print("Extracting endpoints...")
line_indices = []
pt_types     = []
xs = []
ys = []

for idx, row in lines_proj.iterrows():
    geom = row.geometry
    if geom is None or geom.is_empty:
        continue
    coords = list(geom.coords)
    # start
    line_indices.append(idx); pt_types.append('start')
    xs.append(coords[0][0]);  ys.append(coords[0][1])
    # end
    line_indices.append(idx); pt_types.append('end')
    xs.append(coords[-1][0]); ys.append(coords[-1][1])

pts_arr = np.column_stack([xs, ys])   # (N, 2) in EPSG:5070 metres
print(f"  Raw endpoints extracted : {len(pts_arr):,}")

# ── Snap to 500 m grid ────────────────────────────────────────────────────────
# round() to nearest 500m so any two points ≤ 250m from the same cell centre
# are merged.  Real substations are far more than 500m apart in practice.
GRID_SIZE = 500.0  # metres
grid_keys = np.round(pts_arr / GRID_SIZE).astype(np.int64)
unique_keys, inverse_idx = np.unique(grid_keys, axis=0, return_inverse=True)
# inverse_idx[i] = bus_id for the i-th endpoint

n_buses = len(unique_keys)
print(f"  Unique buses after 500 m dedup : {n_buses:,}")

# ── Compute bus centroids (mean of constituent endpoints) ─────────────────────
bus_sum    = np.zeros((n_buses, 2))
bus_counts = np.zeros(n_buses, dtype=np.int64)
np.add.at(bus_sum,    inverse_idx, pts_arr)
np.add.at(bus_counts, inverse_idx, 1)
bus_xy_5070 = bus_sum / bus_counts[:, None]   # (n_buses, 2) in EPSG:5070

# ── Build bus GeoDataFrame ────────────────────────────────────────────────────
bus_gdf = gpd.GeoDataFrame(
    {"bus_id": np.arange(n_buses)},
    geometry=[Point(x, y) for x, y in bus_xy_5070],
    crs="EPSG:5070",
).to_crs(epsg=4326)
bus_gdf["lon"] = bus_gdf.geometry.x
bus_gdf["lat"] = bus_gdf.geometry.y
bus_gdf["bus_id"] = bus_gdf["bus_id"].astype(int)

print(f"  Bus GeoDataFrame built : {len(bus_gdf):,} rows")
print(bus_gdf.head(3).to_string())

# ── Build endpoint → bus_id lookup (vectorised) ───────────────────────────────
# We need: line_idx → {start: bus_id, end: bus_id}
ep_df = pd.DataFrame({
    "line_idx": line_indices,
    "pt_type" : pt_types,
    "bus_id"  : inverse_idx,
})

starts = ep_df[ep_df["pt_type"] == "start"][["line_idx", "bus_id"]].set_index("line_idx")
ends   = ep_df[ep_df["pt_type"] == "end"  ][["line_idx", "bus_id"]].set_index("line_idx")
edge_df = starts.join(ends, lsuffix="_start", rsuffix="_end", how="inner").reset_index()
edge_df.columns = ["line_idx", "src_bus", "dst_bus"]

# Remove self-loops (start and end snapped to same bus)
self_loops = (edge_df["src_bus"] == edge_df["dst_bus"]).sum()
edge_df = edge_df[edge_df["src_bus"] != edge_df["dst_bus"]].copy()
print(f"  Self-loops removed : {self_loops:,}")
print(f"  Valid line→bus edges : {len(edge_df):,}")

────────────────────────────────────────────────────────────
STEP 2 — Build bus topology
────────────────────────────────────────────────────────────
Projecting transmission lines to EPSG:5070...
Extracting endpoints...
  Raw endpoints extracted : 336,050
  Unique buses after 500 m dedup : 61,059
  Bus GeoDataFrame built : 61,059 rows
   bus_id                     geometry         lon        lat
0       0  POINT (-124.11136 40.78249) -124.111359  40.782490
1       1  POINT (-123.78039 40.47998) -123.780395  40.479985
2       2   POINT (-123.77889 40.4825) -123.778885  40.482502
  Self-loops removed : 94,209
  Valid line→bus edges : 73,816


## Step 3 — Snap Plants to Buses

Use `sjoin_nearest` in EPSG:5070 (metres). Plants more than 50 km from
any bus are flagged and excluded from the graph.

In [7]:
print("─" * 60)
print("STEP 3 — Snap plants to buses")
print("─" * 60)

MAX_SNAP_M = 50_000  # 50 km in metres

# Convert capacity to float
plants = plants.copy()
plants["capacity_mw"] = pd.to_numeric(plants["nameplate-capacity-mw"], errors="coerce").fillna(0.0)

# Project both to EPSG:5070 for metric distance
plants_proj  = plants.to_crs(epsg=5070)
bus_gdf_proj = bus_gdf[["bus_id", "geometry"]].to_crs(epsg=5070)

plants_snapped = gpd.sjoin_nearest(
    plants_proj,
    bus_gdf_proj,
    how="left",
    max_distance=MAX_SNAP_M,
    distance_col="snap_dist_m",
)

# Deduplicate: sjoin_nearest can return multiple rows if equidistant — keep closest
plants_snapped = (
    plants_snapped
    .sort_values("snap_dist_m", na_position="last")
    .drop_duplicates(subset=["plantid", "generatorid", "period"], keep="first")
)

excluded = plants_snapped[plants_snapped["bus_id"].isna()]
included = plants_snapped[plants_snapped["bus_id"].notna()].copy()
included["bus_id"] = included["bus_id"].astype(int)

print(f"  Plants snapped to a bus       : {len(included):,}")
print(f"  Plants excluded (> {MAX_SNAP_M/1000:.0f} km)   : {len(excluded):,}")
if len(excluded) > 0:
    print("  Sample excluded plants:")
    print(excluded[["plantName", "stateid", "snap_dist_m"]].head(5).to_string())

print(f"\n  Snap distance stats (included, km):")
d_km = included["snap_dist_m"] / 1000.0
print(f"    mean  : {d_km.mean():.2f}")
print(f"    median: {d_km.median():.2f}")
print(f"    max   : {d_km.max():.2f}")

────────────────────────────────────────────────────────────
STEP 3 — Snap plants to buses
────────────────────────────────────────────────────────────
  Plants snapped to a bus       : 14,354
  Plants excluded (> 50 km)   : 646
  Sample excluded plants:
       plantName stateid  snap_dist_m
0     Sand Point      AK          NaN
1     Sand Point      AK          NaN
2     Sand Point      AK          NaN
129  Annex Creek      AK          NaN
130  Annex Creek      AK          NaN

  Snap distance stats (included, km):
    mean  : 3.58
    median: 0.64
    max   : 49.25


## Step 4 — Build NetworkX Graph

- **Nodes:** one per bus.  Plant-bearing buses get extra attributes.
- **Edges:** one per transmission-line segment (parallel edges between the same
  bus pair are collapsed by `nx.Graph`; last attribute set wins).

In [8]:
print("─" * 60)
print("STEP 4a — Aggregate plants per bus")
print("─" * 60)

# Dominant fuel = fuel type with highest total MW at that bus
fuel_totals = (
    included
    .groupby(["bus_id", "energy-source-desc"])["capacity_mw"]
    .sum()
    .reset_index()
)
dominant_fuel = (
    fuel_totals
    .sort_values("capacity_mw", ascending=False)
    .drop_duplicates("bus_id", keep="first")
    .set_index("bus_id")["energy-source-desc"]
    .rename("dominant_fuel")
)

bus_agg = (
    included
    .groupby("bus_id")
    .agg(plant_count=("plantName", "count"),
         total_capacity_mw=("capacity_mw", "sum"))
    .join(dominant_fuel)
)

print(f"  Buses with snapped plants : {len(bus_agg):,}")
print(f"  Total capacity snapped    : {bus_agg['total_capacity_mw'].sum():,.1f} MW")
print("\n  Top 5 buses by capacity:")
print(bus_agg.sort_values("total_capacity_mw", ascending=False).head(5).to_string())

────────────────────────────────────────────────────────────
STEP 4a — Aggregate plants per bus
────────────────────────────────────────────────────────────
  Buses with snapped plants : 4,244
  Total capacity snapped    : 1,017,080.4 MW

  Top 5 buses by capacity:
        plant_count  total_capacity_mw       dominant_fuel
bus_id                                                    
5860             33             6809.0               Water
46525             4             4658.0             Nuclear
52480            12             4263.0         Natural Gas
6924              4             4224.4             Nuclear
22264             9             4008.4  Subbituminous Coal


In [9]:
print("─" * 60)
print("STEP 4b — Build NetworkX graph")
print("─" * 60)

G = nx.Graph()

# ── Add nodes ─────────────────────────────────────────────────────────────────
print("Adding nodes...")
for row in bus_gdf.itertuples(index=False):
    bid = int(row.bus_id)
    attrs = {"bus_id": bid, "lon": float(row.lon), "lat": float(row.lat)}
    if bid in bus_agg.index:
        r = bus_agg.loc[bid]
        attrs["plant_count"]       = int(r["plant_count"])
        attrs["total_capacity_mw"] = float(r["total_capacity_mw"])
        attrs["dominant_fuel"]     = str(r["dominant_fuel"]) if pd.notna(r["dominant_fuel"]) else ""
    G.add_node(bid, **attrs)

print(f"  Nodes added : {G.number_of_nodes():,}")

# ── Compute line lengths (EPSG:5070) and attach WKT ──────────────────────────
print("Computing edge lengths...")
line_attrs = pd.DataFrame({
    "length_km"   : lines_proj.geometry.length / 1000.0,
    "geometry_wkt": lines.geometry.apply(lambda g: g.wkt),
    "voltage_kv"  : lines["voltage_kv"].fillna(0.0),
}, index=lines.index)

# Merge edge endpoints with line attributes
edge_attrs = edge_df.join(line_attrs, on="line_idx")

# ── Add edges ─────────────────────────────────────────────────────────────────
print("Adding edges...")
edges_added = 0
for row in edge_attrs.itertuples(index=False):
    src = int(row.src_bus)
    dst = int(row.dst_bus)
    G.add_edge(
        src, dst,
        voltage_kv   = float(row.voltage_kv),
        length_km    = round(float(row.length_km), 4),
        geometry_wkt = row.geometry_wkt,
    )
    edges_added += 1

print(f"  Edges added : {G.number_of_edges():,}  (from {edges_added:,} line→bus pairs)")

────────────────────────────────────────────────────────────
STEP 4b — Build NetworkX graph
────────────────────────────────────────────────────────────
Adding nodes...
  Nodes added : 61,059
Computing edge lengths...
Adding edges...
  Edges added : 58,641  (from 73,816 line→bus pairs)


## Step 5 — Diagnostics

In [10]:
components = sorted(nx.connected_components(G), key=len, reverse=True)
print("─" * 60)
print("STEP 5 — Diagnostics")
print("─" * 60)

n_nodes  = G.number_of_nodes()
n_edges  = G.number_of_edges()
n_comps  = nx.number_connected_components(G)
isolated = [n for n, d in G.degree() if d == 0]

print(f"  Total nodes              : {n_nodes:,}")
print(f"  Total edges              : {n_edges:,}")
print(f"  Connected components     : {n_comps:,}")
print(f"  Isolated nodes (deg = 0) : {len(isolated):,}")

# ── Top 10 buses by snapped capacity ─────────────────────────────────────────
print("\n  Top 10 buses by total snapped capacity:")
top10 = bus_agg.sort_values("total_capacity_mw", ascending=False).head(10).copy()
top10 = top10.join(bus_gdf.set_index("bus_id")[["lon", "lat"]])
print(top10.to_string())

# ── Component size distribution ───────────────────────────────────────────────
comp_sizes = sorted([len(c) for c in nx.connected_components(G)], reverse=True)
print(f"\n  Largest component size  : {comp_sizes[0]:,} nodes")
if len(comp_sizes) > 1:
    print(f"  2nd largest             : {comp_sizes[1]:,} nodes")
print(f"  Singletons (size = 1)   : {sum(1 for s in comp_sizes if s == 1):,}")

# ── Warning threshold ─────────────────────────────────────────────────────────
if n_comps > 20:
    print()
    print("  ⚠  WARNING: more than 20 connected components detected.")
    print("  ─────────────────────────────────────────────────────────────")
    print("  Possible causes and fixes:")
    print("   1. Grid cell size (500 m) may be too fine — increase to 1 000 m")
    print("      to merge endpoints at the same substation more aggressively.")
    print("   2. Geographic isolation is real (Alaska, Hawaii, island grids).")
    print("      Inspect small components and exclude non-CONUS buses.")
    print("   3. OSM line data may have gaps — check for T-junction endpoints")
    print("      that are slightly offset. Increase GRID_SIZE in Step 2.")
    print("  ─────────────────────────────────────────────────────────────")
    print("  Proposed next step: re-run Step 2 with GRID_SIZE = 1000.")
else:
    print(f"\n  Connected-component count ({n_comps}) is within acceptable range.")

────────────────────────────────────────────────────────────
STEP 5 — Diagnostics
────────────────────────────────────────────────────────────
  Total nodes              : 61,059
  Total edges              : 58,641
  Connected components     : 8,763
  Isolated nodes (deg = 0) : 3,270

  Top 10 buses by total snapped capacity:
        plant_count  total_capacity_mw       dominant_fuel         lon        lat
bus_id                                                                           
5860             33             6809.0               Water -118.978084  47.956569
46525             4             4658.0             Nuclear  -81.761821  33.144572
52480            12             4263.0         Natural Gas  -80.376109  26.699170
6924              4             4224.4             Nuclear -112.860011  33.386809
22264             9             4008.4  Subbituminous Coal  -95.630104  29.482678
33892             3             3854.0             Nuclear  -87.116894  34.705049
39240           

In [11]:
import networkx as nx

components = sorted(nx.connected_components(G), key=len, reverse=True)
giant = len(components[0])
second = len(components[1]) if len(components) > 1 else 0

print(f"Giant component:  {giant:,} nodes")
print(f"Second component: {second:,} nodes")
print(f"Total components: {len(components):,}")
print(f"Giant as % of all nodes: {giant / G.number_of_nodes() * 100:.1f}%")

# Hard guardrail
if giant > 55000:
    print("\nWARNING: Giant component suspiciously large — possible over-merging")
elif giant < 43000:
    print("\nWARNING: Giant component barely changed — threshold may not have helped")
else:
    print("\nOK: Giant component grew moderately — looks reasonable")

Giant component:  41,766 nodes
Second component: 381 nodes
Total components: 8,763
Giant as % of all nodes: 68.4%



In [12]:
# Rebuild the bus capacity summary from node attributes
bus_rows = []
for node_id, attrs in G.nodes(data=True):
    if attrs.get("total_capacity_mw", 0) > 0:
        bus_rows.append({
            "bus_id":            node_id,
            "total_capacity_mw": attrs["total_capacity_mw"],
            "dominant_fuel":     attrs.get("dominant_fuel", ""),
            "lat":               attrs.get("lat"),
            "lon":               attrs.get("lon"),
        })

bus_df = pd.DataFrame(bus_rows).sort_values("total_capacity_mw", ascending=False)

print("Top 15 buses by snapped capacity:")
print(bus_df.head(15).to_string(index=False))

# Hard guardrail
if bus_df["total_capacity_mw"].max() > 12000:
    print("\nWARNING: Largest bus exceeds 12,000 MW — likely over-merged")

Top 15 buses by snapped capacity:
 bus_id  total_capacity_mw      dominant_fuel       lat         lon
   5860             6809.0              Water 47.956569 -118.978084
  46525             4658.0            Nuclear 33.144572  -81.761821
  52480             4263.0        Natural Gas 26.699170  -80.376109
   6924             4224.4            Nuclear 33.386809 -112.860011
  22264             4008.4 Subbituminous Coal 29.482678  -95.630104
  33892             3854.0            Nuclear 34.705049  -87.116894
  39240             3498.6    Bituminous Coal 34.124012  -84.921701
  45957             3449.0        Natural Gas 28.966994  -82.698948
  32967             3343.5        Natural Gas 31.005700  -88.009793
  32223             3339.5    Bituminous Coal 38.373865  -87.766158
  40150             3293.1 Subbituminous Coal 41.892397  -83.347889
  53558             2978.4            Nuclear 25.436068  -80.332994
  47537             2951.2        Natural Gas 27.604554  -82.346216
  44701       

In [13]:
import collections

size_counts = collections.Counter(len(c) for c in components)

print("Component size distribution:")
print(f"{'Size':>8}  {'Count':>8}  {'Notes'}")
print("-" * 45)
for size in sorted(size_counts.keys()):
    count = size_counts[size]
    note = ""
    if size == 1:
        note = "← true singletons (expect ~3,000-3,500)"
    elif size <= 5:
        note = "← should collapse vs 500m run"
    elif size == len(components[0]):
        note = "← CONUS backbone"
    print(f"{size:>8}  {count:>8}  {note}")

# Summary counts
small = sum(v for k, v in size_counts.items() if k <= 5)
mid   = sum(v for k, v in size_counts.items() if 6 <= k <= 100)
large = sum(v for k, v in size_counts.items() if k > 100)
print(f"\nSmall (≤5 nodes):   {small:,} components")
print(f"Medium (6-100):     {mid:,} components")
print(f"Large (>100):       {large:,} components")

Component size distribution:
    Size     Count  Notes
---------------------------------------------
       1      3270  ← true singletons (expect ~3,000-3,500)
       2      3964  ← should collapse vs 500m run
       3       769  ← should collapse vs 500m run
       4       313  ← should collapse vs 500m run
       5       152  ← should collapse vs 500m run
       6        89  
       7        46  
       8        32  
       9        26  
      10        11  
      11        13  
      12         9  
      13         4  
      14         7  
      15         5  
      16         6  
      17         5  
      18         4  
      19         3  
      20         5  
      21         2  
      22         1  
      23         3  
      24         1  
      26         1  
      27         2  
      28         1  
      30         2  
      32         1  
      35         1  
      39         1  
      40         3  
      41         2  
      43         1  
      45         1  
      49 

In [14]:
from shapely.geometry import Point
import numpy as np

# Sample the size-2 components and measure the distance between their two nodes
size2_components = [c for c in components if len(c) == 2]

distances_m = []
for comp in size2_components[:500]:  # sample first 500
    nodes = list(comp)
    n1 = G.nodes[nodes[0]]
    n2 = G.nodes[nodes[1]]
    p1 = Point(n1['lon'], n1['lat'])
    p2 = Point(n2['lon'], n2['lat'])
    # Approximate metres using degrees → metres at mid-latitude
    dlat = (n1['lat'] - n2['lat']) * 111000
    dlon = (n1['lon'] - n2['lon']) * 111000 * np.cos(np.radians((n1['lat']+n2['lat'])/2))
    distances_m.append(np.sqrt(dlat**2 + dlon**2))

distances_m = np.array(distances_m)
print(f"Size-2 component internal distances:")
print(f"  Median: {np.median(distances_m):.0f} m")
print(f"  75th pct: {np.percentile(distances_m, 75):.0f} m")
print(f"  90th pct: {np.percentile(distances_m, 90):.0f} m")
print(f"  95th pct: {np.percentile(distances_m, 95):.0f} m")
print(f"  Max: {np.max(distances_m):.0f} m")

Size-2 component internal distances:
  Median: 775 m
  75th pct: 2502 m
  90th pct: 6651 m
  95th pct: 12181 m
  Max: 92433 m


In [15]:
# Capacity retained by component size threshold
cap_by_component = {}
for i, comp in enumerate(components):
    total_cap = sum(
        G.nodes[n].get('total_capacity_mw', 0) for n in comp
    )
    cap_by_component[i] = {
        'size': len(comp),
        'total_capacity_mw': total_cap
    }

comp_df = pd.DataFrame(cap_by_component).T.sort_values(
    'total_capacity_mw', ascending=False
)

# Print retention at various size thresholds
total_cap = comp_df['total_capacity_mw'].sum()
print(f"Total capacity in graph: {total_cap:,.0f} MW\n")

for threshold in [1, 10, 50, 100]:
    kept = comp_df[comp_df['size'] >= threshold]['total_capacity_mw'].sum()
    print(f"Components ≥ {threshold:>4} nodes: "
          f"{kept:>10,.0f} MW retained "
          f"({kept/total_cap*100:.1f}%)")

Total capacity in graph: 1,017,080 MW

Components ≥    1 nodes:  1,017,080 MW retained (100.0%)
Components ≥   10 nodes:    892,789 MW retained (87.8%)
Components ≥   50 nodes:    865,775 MW retained (85.1%)
Components ≥  100 nodes:    865,514 MW retained (85.1%)


## Step 6 — Export

In [16]:
print("─" * 60)
print("STEP 6 — Export")
print("─" * 60)

# ── 6a: Verify all node/edge attributes are scalar ───────────────────────────
print("Verifying attribute types...")
SCALAR = (int, float, str, bool)

bad_node_attrs = set()
for _, attrs in G.nodes(data=True):
    for k, v in attrs.items():
        if not isinstance(v, SCALAR):
            bad_node_attrs.add((k, type(v).__name__))

bad_edge_attrs = set()
for _, _, attrs in G.edges(data=True):
    for k, v in attrs.items():
        if not isinstance(v, SCALAR):
            bad_edge_attrs.add((k, type(v).__name__))

if bad_node_attrs:
    print(f"  Non-scalar node attrs: {bad_node_attrs}")
    raise TypeError("Fix non-scalar node attributes before GraphML export.")
if bad_edge_attrs:
    print(f"  Non-scalar edge attrs: {bad_edge_attrs}")
    raise TypeError("Fix non-scalar edge attributes before GraphML export.")
print("  All attributes are scalar types.")

# ── 6b: Save GraphML ──────────────────────────────────────────────────────────
graphml_path = PROCESSED / "grid_network.graphml"
nx.write_graphml(G, str(graphml_path))
size_mb = graphml_path.stat().st_size / 1024**2
print(f"  Saved grid_network.graphml  →  {graphml_path}")
print(f"  File size: {size_mb:.1f} MB")

# ── 6c: Build and save bus_locations.geojson ─────────────────────────────────
# Start from bus_gdf, join plant aggregates where available
bus_out = bus_gdf.copy()
bus_out = bus_out.join(bus_agg, on="bus_id", how="left")
bus_out["plant_count"]       = bus_out["plant_count"].fillna(0).astype(int)
bus_out["total_capacity_mw"] = bus_out["total_capacity_mw"].fillna(0.0)
bus_out["dominant_fuel"]     = bus_out["dominant_fuel"].fillna("").astype(str)

# Add degree from graph
degree_map = dict(G.degree())
bus_out["degree"] = bus_out["bus_id"].map(degree_map).fillna(0).astype(int)

bus_locations_path = PROCESSED / "bus_locations.geojson"
bus_out.to_file(bus_locations_path, driver="GeoJSON")
print(f"  Saved bus_locations.geojson →  {bus_locations_path}")
print(f"  ({len(bus_out):,} buses)")

print("\n  ── COMPLETE ──")
print(f"  Nodes                  : {G.number_of_nodes():,}")
print(f"  Edges                  : {G.number_of_edges():,}")
print(f"  Connected components   : {nx.number_connected_components(G):,}")
print(f"  Buses with plants      : {(bus_out['plant_count'] > 0).sum():,}")
print(f"  Total snapped capacity : {bus_out['total_capacity_mw'].sum():,.1f} MW")

────────────────────────────────────────────────────────────
STEP 6 — Export
────────────────────────────────────────────────────────────
Verifying attribute types...
  All attributes are scalar types.
  Saved grid_network.graphml  →  /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/grid_network.graphml
  File size: 84.0 MB
  Saved bus_locations.geojson →  /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/bus_locations.geojson
  (61,059 buses)

  ── COMPLETE ──
  Nodes                  : 61,059
  Edges                  : 58,641
  Connected components   : 8,763
  Buses with plants      : 4,244
  Total snapped capacity : 1,017,080.4 MW
